# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](https://github.com/mahmoodasayeedi/lab-neural-networks/blob/master/your-code/tttboard.jpg?raw=1)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

In [54]:
!git clone https://github.com/mahmoodasayeedi/lab-neural-networks.git

fatal: destination path 'lab-neural-networks' already exists and is not an empty directory.


Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [55]:
#Load the dataset
import pandas as pd

df = pd.read_csv('/content/lab-neural-networks/your-code/tic-tac-toe.csv')

df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [56]:
#EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB


In [57]:
# Check for missing values to determine reliability
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64


In [58]:
#summarize
df.describe()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
count,958,958,958,958,958,958,958,958,958,958
unique,3,3,3,3,3,3,3,3,3,2
top,x,x,x,x,x,x,x,x,x,True
freq,418,378,418,378,458,378,418,378,418,626


In [59]:
# Separate features (X) and label/target (y)
X = df.iloc[:, :-1].copy()  # Selection of all columns except the last one
y_raw= df.iloc[:, -1].copy()   # Selection of only the last column

print(f"Inputs shape: {X.shape}, Output shape: {y_raw.shape}")

Inputs shape: (958, 9), Output shape: (958,)


In [60]:
#1. Convert the categorical values to numeric in all columns.
from sklearn.preprocessing import LabelEncoder

# Clean up any accidental whitespaces and force lowercase
for col in X.columns:
    X[col] = X[col].astype(str).str.strip().str.lower()

# 3. Clean Feature Mapping (Neutralizes label ordering conflict)
# This maps blank spaces to 0, 'o' to -1, and 'x' to 1
mapping = {'b': 0, 'o': -1, 'x': 1}
for col in X.columns:
    X[col] = X[col].map(mapping)

# Double-check: If anything failed to map, fill it safely with 0
X = X.fillna(0)


# 4. Encode the target column safely (positive -> 1, negative -> 0)
le = LabelEncoder()
y = le.fit_transform(y_raw.astype(str).str.strip().str.lower())

# View data after transformation
print(df.head())



  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True


In [61]:
from sklearn.preprocessing import MinMaxScaler

# 4.Initialize the MinMaxScaler
scaler = MinMaxScaler()

# 5. Normalize features using MinMaxScaler to a clean 0 to 1 scale
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [62]:
# 1. Split the training and test data.
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42
)


In [63]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Initialize the Sequential model
model = Sequential()

# Input layer + first middle (hidden) layer with ReLU
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))

# Additional middle (hidden) layer with ReLU
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
# Output layer with Softmax (assumes 2 unique class labels for the outcome)
model.add(Dense(2, activation='softmax'))



/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
# Compile with a standard stable learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [65]:
# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=120,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)


Epoch 1/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.7141 - loss: 0.5641 - val_accuracy: 0.7532 - val_loss: 0.4890
Epoch 2/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7721 - loss: 0.4878 - val_accuracy: 0.8182 - val_loss: 0.4344
Epoch 3/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8491 - loss: 0.3743 - val_accuracy: 0.8442 - val_loss: 0.3315
Epoch 4/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9115 - loss: 0.2539 - val_accuracy: 0.9351 - val_loss: 0.2465
Epoch 5/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9768 - loss: 0.1338 - val_accuracy: 0.9870 - val_loss: 0.1271
Epoch 6/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9855 - loss: 0.0781 - val_accuracy: 0.9221 - val_loss: 0.1947
Epoch 7/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9768 - loss: 0.0627 - val_accuracy: 0.9610 - val_loss: 0.1072
Epoch 8/120
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9956 - loss: 0.0286 - val_accuracy: 0.9870 - 

In [66]:
# 8. Re-evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\n--- Unified Corrected Model Results ---")
print(f"Final Test Loss:     {test_loss:.4f} (Goal: < 0.1)")
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}% (Goal: > 95%)")


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9896 - loss: 0.0816  

--- Unified Corrected Model Results ---
Final Test Loss:     0.0816 (Goal: < 0.1)
Final Test Accuracy: 98.96% (Goal: > 95%)


In [67]:
# Save the model using the recommended native Keras format
model.save('tic-tac-toe.keras')
print("Model successfully saved as tic-tac-toe.keras")



Model successfully saved as tic-tac-toe.keras


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [68]:
import numpy as np
import pandas as pd
import tensorflow as tf

# 1. Load your saved model
model = tf.keras.models.load_model('tic-tac-toe.keras')

# 2. Select 5 random indices from the test dataset
# (Assumes X_test and y_test are numpy arrays or dataframes from your split step)
np.random.seed(42)  # For reproducible random choices
random_indices = np.random.choice(len(X_test), size=5, replace=False)

# 3. Extract the random samples and their true labels safely
if isinstance(X_test, pd.DataFrame):
    samples = X_test.iloc[random_indices]
else:
    samples = X_test[random_indices]

# Force index matching by using .iloc or converting pandas Series to numpy array
if isinstance(y_test, (pd.Series, pd.DataFrame)):
    true_labels = y_test.iloc[random_indices].values
else:
    true_labels = y_test[random_indices]


# 4. Make predictions
# The model outputs probabilities for each class (e.g., [0.15, 0.85]) due to softmax
raw_predictions = model.predict(samples)

# Use argmax to pick the class index with the highest probability (0 or 1)
predicted_classes = np.argmax(raw_predictions, axis=1)

# 5. Display and compare the results
print("\n--- Prediction Results ---")
for i in range(len(random_indices)):
    print(f"Sample {i+1}:")
    print(f"  Predicted Class: {predicted_classes[i]}")
    print(f"  Actual Class:    {true_labels[i]}")

    # Check if the model got it right
    if predicted_classes[i] == true_labels[i]:
        print("  Result:          ✅ Correct!")
    else:
        print("  Result:          ❌ Incorrect")
    print("-" * 25)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step

--- Prediction Results ---
Sample 1:
  Predicted Class: 0
  Actual Class:    0
  Result:          ✅ Correct!
-------------------------
Sample 2:
  Predicted Class: 1
  Actual Class:    1
  Result:          ✅ Correct!
-------------------------
Sample 3:
  Predicted Class: 1
  Actual Class:    1
  Result:          ✅ Correct!
-------------------------
Sample 4:
  Predicted Class: 1
  Actual Class:    1
  Result:          ✅ Correct!
-------------------------
Sample 5:
  Predicted Class: 0
  Actual Class:    0
  Result:          ✅ Correct!
-------------------------


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [71]:
import tensorflow as tf

# 1. Architecture with Dropout to stabilize and minimize final loss
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.1))  # Prevents final-epoch overfitting fluctuations
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(2, activation='softmax'))

# 2. Optimized compilation
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Train for 150 epochs to guarantee the loss drops below 0.1
history = model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)

# 4. Evaluate the stabilized network
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\n--- Final Stabilized Model Results ---")
print(f"Final Test Loss:     {test_loss:.4f} (Goal: < 0.1)")
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}% (Goal: > 95%)")


Epoch 1/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.6415 - loss: 0.6294 - val_accuracy: 0.7792 - val_loss: 0.5276
Epoch 2/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7199 - loss: 0.5474 - val_accuracy: 0.8052 - val_loss: 0.4705
Epoch 3/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7460 - loss: 0.5014 - val_accuracy: 0.7922 - val_loss: 0.4504
Epoch 4/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7794 - loss: 0.4561 - val_accuracy: 0.8312 - val_loss: 0.4081
Epoch 5/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8186 - loss: 0.3951 - val_accuracy: 0.8571 - val_loss: 0.3673
Epoch 6/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8505 - loss: 0.3515 - val_accuracy: 0.8831 - val_loss: 0.3076
Epoch 7/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9042 - loss: 0.2687 - val_accuracy: 0.8961 - val_loss: 0.2582
Epoch 8/150
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9347 - loss: 0.2191 - val_accuracy: 0.9091 - 

**Which approach(es) did you find helpful to improve your model performance?**

In [74]:
# Save your optimized model to the native Keras format
model.save('tic-tac-toe_optimized.keras')
print("Optimized model successfully saved as 'tic-tac-toe_optimized.keras'!")


Optimized model successfully saved as 'tic-tac-toe_optimized.keras'!


In [73]:
# ==============================================================================
# OPTIMIZATION SUMMARY & MODEL PERFORMANCE NOTES:
#
# 1. Spatial Feature Mapping: Replaced default LabelEncoder with a custom
#    map ('b':0, 'o':-1, 'x':1). This eliminated unintended numerical hierarchies.
# 2. Capacity Adjustment: Expanded hidden layout to 128 -> 64 -> 32 nodes
#    to successfully capture intersecting horizontal, vertical, and diagonal lines.
# 3. Regularization: Added a 10% Dropout layer and tracked for 150 epochs
#    to curb late-stage overfitting fluctuations on small tabular structures.
#
# Target Benchmarks Met: Final Test Loss: 0.0449 | Final Test Accuracy: 97.92%
# ==============================================================================